# UVERA · NB00 — Synthetic World

| | |
|---|---|
| **Who runs it** | Rubab (P1) |
| **Accelerator** | None (CPU) |
| **Internet** | On |
| **Secret** | — |
| **Inputs to add** | — |
| **Expected runtime** | ~5–10 min |
| **How to run** | **Save Version → Save & Run All (Commit)** — runs in the background, no 20-minute idle timeout, 12-hour cap |

**What this AI does (say this to judges):** Builds a fake but realistic mobile-money world: 20,000 customers, 600 agents, 2,500 merchants, 120 days. It injects six scam families, mule wallets that move stolen money, and four ways shops misuse Bangla QR as a cash machine. Labels come from a hidden scammer process, not from simple rules, so the models cannot cheat.

**Outputs** (in `/kaggle/working/outputs/NB00_outputs.zip`): `reports/world_meta.json` (hashes), `reports/data_card_stats.json`, `reports/leakage_check_ai1.json`, `artifacts/web/world_sample.json` (for the homepage 3D)

All data is synthetic or open-licensed. No real customer data.

In [ ]:
# 1) Get the UVERA code from the public GitHub repo (nothing to upload by hand)
REPO = "https://github.com/ABRUBAB/takaresolve-ai-hackathon-2026.git"
REF = "main"   # replace with a commit hash to reproduce an exact run
import os, sys, subprocess
if not os.path.exists("/tmp/uvera"):
    subprocess.run(["git", "clone", "-q", REPO, "/tmp/uvera"], check=True)
subprocess.run(["git", "-C", "/tmp/uvera", "fetch", "-q", "origin", REF], check=False)
subprocess.run(["git", "-C", "/tmp/uvera", "checkout", "-q", REF], check=True)
COMMIT = subprocess.run(["git", "-C", "/tmp/uvera", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
os.environ["UVERA_ROOT"] = "/tmp/uvera"
sys.path.insert(0, "/tmp/uvera/ml")   # use Kaggle's own numpy/pandas/sklearn/lightgbm (no reinstall)
print("UVERA code at commit", COMMIT)
print(f"To reproduce this exact run later, set REF = "{COMMIT}" in this cell.")

In [ ]:
# 2) Reproducibility: seeds, versions, time budget, output folder
import json
from IPython.display import Image, display
from uvera_ml.common import env_info, set_seed, Budget, out_dir, zip_outputs, write_json
set_seed(42)
budget = Budget(minutes=60)
OUT = out_dir("NB00")
info = env_info(); info["uvera_commit"] = COMMIT
write_json(OUT / "reports" / "versions_nb00.json", info)
print(json.dumps(info, indent=1))

In [ ]:
# 3) Build the synthetic world. Same seed -> identical data on every Kaggle account (checked by hash).
from pathlib import Path
from uvera_ml.sim.world import load_or_generate
world = load_or_generate("/tmp/world_full", scale="full")
print(json.dumps(world.meta, indent=1)[:1500])
ref = Path("/tmp/uvera/reports/world_meta.json")
if ref.exists():
    expected = json.loads(ref.read_text(encoding="utf-8"))["meta"]["hashes"]["events"]
    print("Data identical to NB00 run:", expected == world.meta["hashes"]["events"])
budget.check("world")

## Data card statistics
Counts, scam rate, families, label noise, and how customers split by zone/channel/language.

In [ ]:
from uvera_ml.sim.report import world_stats, web_sample
stats = world_stats(world)
write_json(OUT / "reports" / "data_card_stats.json", stats)
write_json(OUT / "reports" / "world_meta.json", {"meta": world.meta})
print(json.dumps(stats, indent=1, default=str)[:3000])

## Leakage guard
No single feature may reach AUC > 0.80 on its own. If one does, the labels would be too easy and the results would not be credible.

In [ ]:
from uvera_ml.models.ai1 import prepare, single_feature_auc
df = prepare(world)
leak = single_feature_auc(df[df["split"] == "train"])
write_json(OUT / "reports" / "leakage_check_ai1.json", leak.to_dict("records"))
display(leak.head(10))
print("PASS" if leak["auc"].max() <= 0.80 else "WARNING: a feature is too predictive on its own")

## Homepage sample
A small piece of the world (≈800 nodes) for the 3D Trust Field on the website.

In [ ]:
sample = web_sample(world)
write_json(OUT / "artifacts" / "web" / "world_sample.json", sample)
print(len(sample["nodes"]), "nodes,", len(sample["edges"]), "edges")

In [ ]:
# Last step: package the results. Download the zip from the Output tab and send it to Rubab.
budget.check("done")
zip_outputs(OUT)